# Prophet Ablation — GB-Only Run
**Purpose:** Remove Prophet from the ensemble and compare RMSLE against the hybrid.
This is a direct ablation test to quantify Prophet's contribution.

| Config | Description |
|--------|-------------|
| Hybrid (baseline) | 1yr GB + 2yr Prophet, per-zone weights |
| GB-only (this run) | 1yr GB predictions only, no blending |

## Cell 1: Configuration

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, mean_squared_log_error

# ── Paths — point to your correctReleases folder ──────────────
BASE_DIR  = Path('.')                        
DIR_1YR   = BASE_DIR / 'run1_1year'
DIR_2YR   = BASE_DIR / 'run2_2year'
DIR_HYB   = BASE_DIR / 'run_hybrid'
ABLATION  = BASE_DIR / 'run_ablation_gb_only'
SELECTED  = BASE_DIR / 'selected_buildings.csv'

for d in [ABLATION / 'results', ABLATION / 'plots']:
    Path(d).mkdir(parents=True, exist_ok=True)

print('✓ Paths configured')
print(f'  GB predictions : {DIR_1YR}/predictions/gb_predictions.parquet')
print(f'  Prophet preds  : {DIR_2YR}/predictions/prophet_predictions.parquet')
print(f'  Hybrid preds   : {DIR_HYB}/predictions/hybrid_predictions.parquet')
print(f'  Output         : {ABLATION}')

✓ Paths configured
  GB predictions : run1_1year/predictions/gb_predictions.parquet
  Prophet preds  : run2_2year/predictions/prophet_predictions.parquet
  Hybrid preds   : run_hybrid/predictions/hybrid_predictions.parquet
  Output         : run_ablation_gb_only


In [ ]:
# Check what's actually inside run_hybrid
import os
for root, dirs, files in os.walk('run_hybrid'):
    for f in files:
        print(os.path.join(root, f))

## Cell 2: Metrics Helper

In [2]:
def compute_metrics(y_true, y_pred, label=''):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    mask   = y_true > 0
    yt     = y_true[mask]
    yp     = y_pred[mask].clip(min=0.001)
    mape_m = yt > 0.5
    return {
        'model' : label,
        'rmsle' : np.sqrt(mean_squared_log_error(yt, yp)),
        'rmse'  : np.sqrt(mean_squared_error(yt, yp)),
        'mae'   : mean_absolute_error(yt, yp),
        'r2'    : r2_score(yt, yp),
        'mape'  : np.mean(np.abs((yt[mape_m]-yp[mape_m])/yt[mape_m]))*100
                  if mape_m.sum() > 0 else np.nan,
        'n'     : int(mask.sum())
    }

def print_table(metrics_list):
    print(f'\n{"Model":<28} {"RMSLE":>8} {"RMSE":>8} {"MAE":>8} {"R²":>8} {"MAPE%":>8}')
    print('-'*76)
    for m in metrics_list:
        print(f'  {m["model"]:<26} {m["rmsle"]:>8.4f} {m["rmse"]:>8.4f} '
              f'{m["mae"]:>8.4f} {m["r2"]:>8.4f} {m["mape"]:>8.2f}')

print('✓ Metrics helpers ready')

✓ Metrics helpers ready


## Cell 3: Load All Predictions
Load GB, Prophet, and hybrid ensemble predictions — all on the same Oct–Dec 2018 test set.

In [3]:
print('Loading predictions...')

gb_df      = pd.read_parquet(DIR_1YR / 'predictions' / 'gb_predictions.parquet')
prophet_df = pd.read_parquet(DIR_2YR / 'predictions' / 'prophet_predictions.parquet')
hybrid_df  = pd.read_parquet(DIR_HYB / 'predictions' / 'hybrid_predictions.parquet')

# Normalise building_id dtype
for df in [gb_df, prophet_df, hybrid_df]:
    df['building_id'] = df['building_id'].astype(str)

print(f'  GB predictions  : {gb_df.shape}  | buildings: {gb_df["building_id"].nunique()}')
print(f'  Prophet preds   : {prophet_df.shape}  | buildings: {prophet_df["building_id"].nunique()}')
print(f'  Hybrid ensemble : {hybrid_df.shape}  | buildings: {hybrid_df["building_id"].nunique()}')
print(f'\n  Test period: {gb_df["timestamp"].min()} → {gb_df["timestamp"].max()}')
print(f'  Ground truth mean: {gb_df["total_kwh"].mean():.4f} kWh')

Loading predictions...
  GB predictions  : (220900, 75)  | buildings: 100
  Prophet preds   : (220900, 26)  | buildings: 100
  Hybrid ensemble : (220900, 13)  | buildings: 100

  Test period: 2018-10-01 00:00:00 → 2019-01-01 00:00:00
  Ground truth mean: 2.4464 kWh


## Cell 4: GB-Only vs Hybrid — Overall Comparison
This is the core ablation. GB-only = Prophet weight forced to 0 across all zones.

In [5]:
# GB-only — no blending at all
m_gb     = compute_metrics(gb_df['total_kwh'],     gb_df['gb_prediction'],           'GB only (no Prophet)')
m_prophet= compute_metrics(prophet_df['total_kwh'],prophet_df['prophet_prediction'],  'Prophet only')
m_hybrid = compute_metrics(hybrid_df['total_kwh'], hybrid_df['hybrid_prediction'],  'Hybrid (GB + Prophet)')

print('='*76)
print('ABLATION RESULT: Prophet contribution to ensemble')
print('='*76)
print_table([m_gb, m_prophet, m_hybrid])

# Key delta — how much does Prophet blending actually help?
rmsle_delta = m_gb['rmsle'] - m_hybrid['rmsle']
rmse_delta  = m_gb['rmse']  - m_hybrid['rmse']
r2_delta    = m_hybrid['r2']- m_gb['r2']

print(f'\nProphet contribution (Hybrid minus GB-only):')
print(f'  RMSLE improvement : {rmsle_delta:+.4f}  '
      f'({rmsle_delta/m_gb["rmsle"]*100:+.2f}%)  '
      f'{"✓ Prophet helps" if rmsle_delta > 0 else "✗ Prophet hurts or neutral"}')
print(f'  RMSE  improvement : {rmse_delta:+.4f} kWh')
print(f'  R²    improvement : {r2_delta:+.4f}')

# Save
pd.DataFrame([m_gb, m_prophet, m_hybrid]).to_csv(
    ABLATION / 'results' / 'ablation_overall.csv', index=False)
print(f'\n✓ Saved: {ABLATION}/results/ablation_overall.csv')

ABLATION RESULT: Prophet contribution to ensemble

Model                           RMSLE     RMSE      MAE       R²    MAPE%
----------------------------------------------------------------------------
  GB only (no Prophet)         0.1914   1.0020   0.4879   0.8630    21.76
  Prophet only                 0.3240   1.7733   0.8792   0.5709    38.91
  Hybrid (GB + Prophet)        0.1905   1.0041   0.4891   0.8624    21.85

Prophet contribution (Hybrid minus GB-only):
  RMSLE improvement : +0.0009  (+0.49%)  ✓ Prophet helps
  RMSE  improvement : -0.0021 kWh
  R²    improvement : -0.0006

✓ Saved: run_ablation_gb_only/results/ablation_overall.csv


## Cell 5: Per-Zone Ablation
Prophet weight varies by zone (2.5% to 10%). Check where it helps and where it doesn't.

In [6]:
# Load known weights
weights = pd.read_csv(SELECTED.parent / 'ensemble_weights.csv') \
    if (SELECTED.parent / 'ensemble_weights.csv').exists() \
    else pd.DataFrame({
        'zone'   : ['2B','3B','3C','4B','4C','5B'],
        'prophet': [0.075,0.050,0.075,0.025,0.075,0.100],
        'gb'     : [0.925,0.950,0.925,0.975,0.925,0.900]
    })

# Merge zone onto gb_df
zone_map = pd.read_csv(SELECTED).set_index('bid_str')['climate_zone'].to_dict()
gb_df['climate_zone'] = gb_df['building_id'].map({str(k):v for k,v in zone_map.items()})

# Merge prophet predictions onto gb_df for per-zone blending
merged = gb_df[['building_id','timestamp','total_kwh',
                'climate_zone','gb_prediction']].merge(
    prophet_df[['building_id','timestamp','prophet_prediction']],
    on=['building_id','timestamp'], how='inner'
)

zone_results = []
print(f'\n{"Zone":<6} {"Prophet W":>10} {"GB-only RMSLE":>14} '
      f'{"Hybrid RMSLE":>14} {"Delta":>10} {"Verdict":>18}')
print('-'*78)

for zone in sorted(merged['climate_zone'].dropna().unique()):
    z    = merged[merged['climate_zone'] == zone]
    zw   = weights[weights['zone']==zone]
    w_p  = float(zw['prophet'].values[0]) if len(zw) > 0 else 0.0
    w_gb = 1.0 - w_p

    # GB-only for this zone
    m_z_gb  = compute_metrics(z['total_kwh'], z['gb_prediction'],      f'GB {zone}')

    # Hybrid for this zone
    hybrid_pred = (w_p * z['prophet_prediction'] +
                   w_gb * z['gb_prediction']).clip(lower=0.001)
    m_z_hyb = compute_metrics(z['total_kwh'], hybrid_pred, f'Hybrid {zone}')

    delta   = m_z_gb['rmsle'] - m_z_hyb['rmsle']
    verdict = '✓ Prophet helps' if delta > 0.001 else \
              '✗ Prophet hurts' if delta < -0.001 else \
              '~ Negligible'

    print(f'  {zone:<4}  {w_p:>10.3f}  {m_z_gb["rmsle"]:>14.4f}  '
          f'{m_z_hyb["rmsle"]:>14.4f}  {delta:>+10.4f}  {verdict:>18}')

    zone_results.append({
        'zone': zone, 'prophet_weight': w_p,
        'gb_only_rmsle': m_z_gb['rmsle'],
        'hybrid_rmsle' : m_z_hyb['rmsle'],
        'delta'        : delta, 'verdict': verdict
    })

zone_df = pd.DataFrame(zone_results)
zone_df.to_csv(ABLATION / 'results' / 'ablation_per_zone.csv', index=False)
print(f'\n✓ Saved: {ABLATION}/results/ablation_per_zone.csv')


Zone    Prophet W  GB-only RMSLE   Hybrid RMSLE      Delta            Verdict
------------------------------------------------------------------------------
  2B         0.075          0.1677          0.1662     +0.0015     ✓ Prophet helps
  3B         0.050          0.1801          0.1792     +0.0008        ~ Negligible
  3C         0.075          0.2053          0.2042     +0.0012     ✓ Prophet helps
  4B         0.025          0.2079          0.2078     +0.0001        ~ Negligible
  4C         0.075          0.1797          0.1786     +0.0011     ✓ Prophet helps
  5B         0.100          0.1904          0.1888     +0.0017     ✓ Prophet helps

✓ Saved: run_ablation_gb_only/results/ablation_per_zone.csv


## Cell 6: Temporal Analysis — Where Does Prophet Help?
Check if Prophet's contribution varies by month or time-of-day within the test window.

In [7]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

merged['month']    = pd.to_datetime(merged['timestamp']).dt.month
merged['hour']     = pd.to_datetime(merged['timestamp']).dt.hour
merged['w_p']      = merged['climate_zone'].map(
    weights.set_index('zone')['prophet'].to_dict()).fillna(0)
merged['hybrid_pred'] = (
    merged['w_p'] * merged['prophet_prediction'] +
    (1 - merged['w_p']) * merged['gb_prediction']
).clip(lower=0.001)

clean = merged[merged['total_kwh'] > 0].copy()

# Per-month RMSLE
month_results = []
for month in sorted(clean['month'].unique()):
    m_data = clean[clean['month'] == month]
    gb_r   = np.sqrt(mean_squared_log_error(
        m_data['total_kwh'], m_data['gb_prediction'].clip(0.001)))
    hyb_r  = np.sqrt(mean_squared_log_error(
        m_data['total_kwh'], m_data['hybrid_pred'].clip(0.001)))
    month_results.append({'month': month, 'gb_rmsle': gb_r,
                          'hybrid_rmsle': hyb_r, 'delta': gb_r - hyb_r})
month_df = pd.DataFrame(month_results)

# Per-hour RMSLE
hour_results = []
for hour in range(24):
    h_data = clean[clean['hour'] == hour]
    gb_r   = np.sqrt(mean_squared_log_error(
        h_data['total_kwh'], h_data['gb_prediction'].clip(0.001)))
    hyb_r  = np.sqrt(mean_squared_log_error(
        h_data['total_kwh'], h_data['hybrid_pred'].clip(0.001)))
    hour_results.append({'hour': hour, 'gb_rmsle': gb_r,
                         'hybrid_rmsle': hyb_r, 'delta': gb_r - hyb_r})
hour_df = pd.DataFrame(hour_results)

# Plot
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
month_names = {10:'Oct', 11:'Nov', 12:'Dec'}

# Monthly
x = range(len(month_df))
w = 0.35
axes[0].bar([i-w/2 for i in x], month_df['gb_rmsle'],
            width=w, label='GB only', color='#1D9E75', alpha=0.85)
axes[0].bar([i+w/2 for i in x], month_df['hybrid_rmsle'],
            width=w, label='Hybrid', color='#E07B39', alpha=0.85)
axes[0].set_xticks(list(x))
axes[0].set_xticklabels([month_names.get(m, str(m)) for m in month_df['month']])
axes[0].set_ylabel('RMSLE')
axes[0].set_title('Monthly RMSLE — GB-only vs Hybrid', fontweight='bold')
axes[0].legend(fontsize=9)
axes[0].spines['top'].set_visible(False)
axes[0].spines['right'].set_visible(False)

# Hourly delta (positive = hybrid better)
colors = ['#1D9E75' if d >= 0 else '#E24B4A' for d in hour_df['delta']]
axes[1].bar(hour_df['hour'], hour_df['delta'], color=colors, alpha=0.85)
axes[1].axhline(0, color='gray', linewidth=0.8)
axes[1].set_xlabel('Hour of day')
axes[1].set_ylabel('RMSLE delta (GB − Hybrid)')
axes[1].set_title('Hourly: where Prophet adds vs hurts\n(green = hybrid better, red = GB better)',
                  fontweight='bold')
axes[1].spines['top'].set_visible(False)
axes[1].spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig(ABLATION / 'plots' / 'temporal_ablation.png', dpi=150, bbox_inches='tight')
plt.close()
print('✓ Figure saved: temporal_ablation.png')
print(f'\nMonthly breakdown:')
print(month_df.to_string(index=False))
print(f'\nHours where Prophet helps (delta > 0): {(hour_df["delta"]>0.001).sum()}/24')
print(f'Hours where Prophet hurts (delta < 0): {(hour_df["delta"]<-0.001).sum()}/24')

✓ Figure saved: temporal_ablation.png

Monthly breakdown:
 month  gb_rmsle  hybrid_rmsle     delta
     1  0.701329      0.707998 -0.006670
    10  0.179608      0.178614  0.000993
    11  0.191489      0.189819  0.001671
    12  0.200902      0.200628  0.000275

Hours where Prophet helps (delta > 0): 13/24
Hours where Prophet hurts (delta < 0): 3/24


## Cell 7: Final Summary for Thesis Reporting

In [8]:
print('='*76)
print('ABLATION SUMMARY — FOR THESIS REPORTING')
print('='*76)

overall_delta  = m_gb['rmsle'] - m_hybrid['rmsle']
pct_change     = overall_delta / m_gb['rmsle'] * 100
zones_helped   = zone_df[zone_df['delta'] >  0.001]
zones_hurt     = zone_df[zone_df['delta'] < -0.001]
zones_neutral  = zone_df[abs(zone_df['delta']) <= 0.001]

print(f'\nOverall:')
print(f'  GB-only RMSLE       : {m_gb["rmsle"]:.4f}')
print(f'  Hybrid RMSLE        : {m_hybrid["rmsle"]:.4f}')
print(f'  Prophet contribution: {overall_delta:+.4f}  ({pct_change:+.2f}%)')

if overall_delta > 0.002:
    verdict = 'Prophet provides a statistically meaningful improvement. Retain in ensemble.'
elif overall_delta > 0:
    verdict = 'Prophet provides marginal improvement (<0.2%). Contribution is minimal but non-negative.'
elif overall_delta > -0.002:
    verdict = 'Prophet is effectively neutral. GB alone matches the hybrid ensemble.'
else:
    verdict = 'Prophet is detrimental. GB-only outperforms the hybrid ensemble.'

print(f'\nVerdict: {verdict}')

print(f'\nPer-zone breakdown:')
print(f'  Zones where Prophet helps   : {list(zones_helped["zone"])}')
print(f'  Zones where Prophet hurts   : {list(zones_hurt["zone"])}')
print(f'  Zones where Prophet neutral : {list(zones_neutral["zone"])}')

print(f'\nThesis recommendation:')
if overall_delta <= 0:
    print('  Report GB-only as the final model.')
    print('  Include Prophet ablation as a finding: hybrid blending adds no value')
    print('  when GB already captures diurnal/weekly seasonality via lag features.')
else:
    print('  Retain hybrid ensemble. Report Prophet contribution as marginal but positive.')
    print('  Note low weights (2.5–10%) reflect GB dominance of seasonal feature space.')

# Save full summary
summary = {
    'gb_only_rmsle'  : m_gb['rmsle'],
    'hybrid_rmsle'   : m_hybrid['rmsle'],
    'prophet_rmsle'  : m_prophet['rmsle'],
    'prophet_delta'  : overall_delta,
    'prophet_pct'    : pct_change,
    'verdict'        : verdict,
}
pd.Series(summary).to_csv(ABLATION / 'results' / 'ablation_summary.csv')
print(f'\n✓ Full summary saved: {ABLATION}/results/ablation_summary.csv')

ABLATION SUMMARY — FOR THESIS REPORTING

Overall:
  GB-only RMSLE       : 0.1914
  Hybrid RMSLE        : 0.1905
  Prophet contribution: +0.0009  (+0.49%)

Verdict: Prophet provides marginal improvement (<0.2%). Contribution is minimal but non-negative.

Per-zone breakdown:
  Zones where Prophet helps   : ['2B', '3C', '4C', '5B']
  Zones where Prophet hurts   : []
  Zones where Prophet neutral : ['3B', '4B']

Thesis recommendation:
  Retain hybrid ensemble. Report Prophet contribution as marginal but positive.
  Note low weights (2.5–10%) reflect GB dominance of seasonal feature space.

✓ Full summary saved: run_ablation_gb_only/results/ablation_summary.csv
